In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from spectral import open_image

In [ ]:
hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"

img = open_image(hdr_path)
cube = np.asarray(img.load(), dtype=np.float32)

wavelengths = np.array([float(w) for w in img.metadata["wavelength"]], dtype=np.float32)

print("cube shape:", cube.shape)
print("wavelengths shape:", wavelengths.shape)
print("primeras longitudes de onda:", wavelengths[:10])

In [ ]:
# Función para encontrar el índice de la banda más cercana a una longitud de onda objetivo
def find_nearest_band(wavelengths, target_nm):
    wavelengths = np.asarray(wavelengths, dtype=float)
    idx = np.argmin(np.abs(wavelengths - target_nm))
    return idx, wavelengths[idx]

# Función de normalización robusta
def robust_normalize(channel, p_low=2, p_high=98):
    ch = channel.astype(np.float32)
    ch = np.where(np.isfinite(ch), ch, np.nan)

    if np.all(np.isnan(ch)):
        return np.zeros_like(ch, dtype=np.float32)

    lo = np.nanpercentile(ch, p_low)
    hi = np.nanpercentile(ch, p_high)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)

    ch = (ch - lo) / (hi - lo)
    ch = np.clip(ch, 0, 1)
    ch = np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0)

    return ch.astype(np.float32)
# Función para convertir a pseudo-RGB
def hsi_to_pseudo_rgb(cube, wavelengths, r_nm=650, g_nm=550, b_nm=450):
    r_idx, r_real = find_nearest_band(wavelengths, r_nm)
    g_idx, g_real = find_nearest_band(wavelengths, g_nm)
    b_idx, b_real = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    rgb = np.stack([r, g, b], axis=-1)

    info = {
        "r_idx": r_idx, "r_nm": r_real,
        "g_idx": g_idx, "g_nm": g_real,
        "b_idx": b_idx, "b_nm": b_real
    }
    return rgb, info

In [ ]:
pseudo_rgb, rgb_info = hsi_to_pseudo_rgb(cube, wavelengths)

print(rgb_info)

plt.figure(figsize=(6, 6))
plt.imshow(pseudo_rgb)
plt.title(
    f"Pseudo-RGB | "
    f"R={rgb_info['r_nm']:.1f}nm, "
    f"G={rgb_info['g_nm']:.1f}nm, "
    f"B={rgb_info['b_nm']:.1f}nm"
)
plt.axis("off")
plt.show()

Segmentación

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Convertir pseudo_rgb a uint8
# -----------------------------
pseudo_rgb = np.nan_to_num(pseudo_rgb, nan=0.0, posinf=1.0, neginf=0.0)
rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)
hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

# -----------------------------
# 2. Máscara de caja azul
# -----------------------------
lower_blue = np.array([85, 10, 40], dtype=np.uint8)
upper_blue = np.array([125, 180, 255], dtype=np.uint8)

mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))

mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
mask_blue = cv2.dilate(mask_blue, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)), iterations=1)

# Componente azul más grande
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_blue, connectivity=8)

areas = stats[1:, cv2.CC_STAT_AREA]
largest_label = 1 + np.argmax(areas)

mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

# Bounding box de la caja
x = stats[largest_label, cv2.CC_STAT_LEFT]
y = stats[largest_label, cv2.CC_STAT_TOP]
w = stats[largest_label, cv2.CC_STAT_WIDTH]
h = stats[largest_label, cv2.CC_STAT_HEIGHT]

pad = 10
H_img, W_img = rgb_u8.shape[:2]

x1 = max(x - pad, 0)
y1 = max(y - pad, 0)
x2 = min(x + w + pad, W_img)
y2 = min(y + h + pad, H_img)

# Recorte RGB y recorte HSI
crop_rgb = rgb_u8[y1:y2, x1:x2]
crop_cube = cube[y1:y2, x1:x2, :]

# -----------------------------
# 3. Segmentación del espécimen dentro del recorte
# -----------------------------
crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)

H = crop_hsv[:, :, 0]
S = crop_hsv[:, :, 1]
V = crop_hsv[:, :, 2]

R = crop_rgb[:, :, 0].astype(np.int16)
G = crop_rgb[:, :, 1].astype(np.int16)
B = crop_rgb[:, :, 2].astype(np.int16)

# Tejido: rojo/marrón, saturado, no muy claro
mask_red_brown = (
    ((H < 35) | (H > 165)) &
    (S > 35) &
    (V > 20) &
    (V < 240) &
    (R > G + 5) &
    (R > B + 5)
)

mask_specimen = mask_red_brown.astype(np.uint8) * 255

# Limpieza morfológica
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

mask_specimen = cv2.morphologyEx(mask_specimen, cv2.MORPH_OPEN, kernel)
mask_specimen = cv2.morphologyEx(mask_specimen, cv2.MORPH_CLOSE, kernel)

# Rellenar huecos internos
contours, _ = cv2.findContours(mask_specimen, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if len(contours) == 0:
    raise ValueError("No se detectó el espécimen. Hay que ajustar los umbrales.")

# Quedarse con el contorno más grande
largest_contour = max(contours, key=cv2.contourArea)

mask_specimen_clean = np.zeros(mask_specimen.shape, dtype=np.uint8)
cv2.drawContours(mask_specimen_clean, [largest_contour], -1, 255, thickness=cv2.FILLED)

# Suavizar un poco el borde
mask_specimen_clean = cv2.morphologyEx(
    mask_specimen_clean,
    cv2.MORPH_CLOSE,
    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
)

# Recalcular contorno final
contours, _ = cv2.findContours(mask_specimen_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
specimen_contour = max(contours, key=cv2.contourArea)

# -----------------------------
# 4. Aplicar máscara al cubo HSI recortado
# -----------------------------
specimen_bool = mask_specimen_clean > 0

specimen_cube = crop_cube.copy()
specimen_cube[~specimen_bool, :] = np.nan

spectra_specimen = crop_cube[specimen_bool, :]

print("crop_rgb shape:", crop_rgb.shape)
print("crop_cube shape:", crop_cube.shape)
print("specimen pixels:", spectra_specimen.shape[0])
print("spectra shape:", spectra_specimen.shape)

# -----------------------------
# 5. Visualización
# -----------------------------
overlay = crop_rgb.copy()
cv2.drawContours(overlay, [specimen_contour], -1, (0, 255, 0), 2)

masked_vis = crop_rgb.copy()
masked_vis[~specimen_bool] = 0

plt.figure(figsize=(16, 5))

plt.subplot(1, 4, 1)
plt.imshow(rgb_u8)
plt.title("Pseudo-RGB original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(crop_rgb)
plt.title("Caja azul + contenido")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(mask_specimen_clean, cmap="gray")
plt.title("Máscara espécimen")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(overlay)
plt.title("Contorno del espécimen")
plt.axis("off")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# pseudo_rgb viene en RGB float [0, 1]
pseudo_rgb = np.nan_to_num(pseudo_rgb, nan=0.0, posinf=1.0, neginf=0.0)
rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)

# Convertir RGB -> HSV
hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

# Separar canales RGB y HSV
R = rgb_u8[:, :, 0].astype(np.int16)
G = rgb_u8[:, :, 1].astype(np.int16)
B = rgb_u8[:, :, 2].astype(np.int16)

H = hsv[:, :, 0]
S = hsv[:, :, 1]
V = hsv[:, :, 2]

# Máscara inicial: tonos azul/cian claros
# OpenCV usa H en [0, 179]
lower_blue = np.array([85, 10, 40], dtype=np.uint8)
upper_blue = np.array([125, 180, 255], dtype=np.uint8)

mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)

# Condición adicional: el azul debe dominar sobre el rojo
mask_dominance = ((B - R) > 8) & ((G - R) > 3) & (V > 50)

mask_blue = mask_hsv & mask_dominance.astype(np.uint8) * 255

# Limpieza morfológica
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))

mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)

# Dilatar para conectar toda la caja azul, incluida la rejilla
kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

# Comprobar que se ha detectado algo
if np.count_nonzero(mask_blue) == 0:
    raise ValueError("No se detectó la caja azul. Hay que relajar los umbrales HSV.")

# Quedarse con el componente conectado más grande
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_blue, connectivity=8)

# Ignorar fondo: label 0
areas = stats[1:, cv2.CC_STAT_AREA]
largest_label = 1 + np.argmax(areas)

mask_box = (labels == largest_label).astype(np.uint8) * 255

# Bounding box de la caja azul
x = stats[largest_label, cv2.CC_STAT_LEFT]
y = stats[largest_label, cv2.CC_STAT_TOP]
w = stats[largest_label, cv2.CC_STAT_WIDTH]
h = stats[largest_label, cv2.CC_STAT_HEIGHT]

# Añadir margen
pad = 10
H_img, W_img = rgb_u8.shape[:2]

x1 = max(x - pad, 0)
y1 = max(y - pad, 0)
x2 = min(x + w + pad, W_img)
y2 = min(y + h + pad, H_img)

# Recorte de la imagen pseudo-RGB
crop_rgb = rgb_u8[y1:y2, x1:x2]

# Recorte equivalente del cubo hiperespectral
crop_cube = cube[y1:y2, x1:x2, :]

print("BBox:", x1, y1, x2, y2)
print("crop_rgb shape:", crop_rgb.shape)
print("crop_cube shape:", crop_cube.shape)

# Visualización
vis = rgb_u8.copy()
cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(rgb_u8)
plt.title("Pseudo-RGB original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask_box, cmap="gray")
plt.title("Máscara caja azul")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(crop_rgb)
plt.title("Recorte: caja azul + contenido")
plt.axis("off")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Usa la máscara limpia de la caja azul
# Debe ser uint8 con valores 0 y 255
mask_box = mask_box_blue.copy()
mask_box = mask_box.astype(np.uint8)

# Buscar contornos con jerarquía
contours, hierarchy = cv2.findContours(
    mask_box,
    cv2.RETR_CCOMP,      # detecta contornos externos e internos
    cv2.CHAIN_APPROX_SIMPLE
)

if hierarchy is None:
    raise ValueError("No se encontraron contornos en la máscara de la caja.")

hierarchy = hierarchy[0]

# Los contornos internos tienen parent != -1
inner_contours = [
    cnt for cnt, h in zip(contours, hierarchy)
    if h[3] != -1
]

if len(inner_contours) == 0:
    raise ValueError("No se encontró ningún contorno interno. La máscara azul no tiene hueco.")

# El hueco interno más grande será el espécimen
specimen_contour = max(inner_contours, key=cv2.contourArea)

# Crear máscara del espécimen rellenando ese contorno interno
mask_specimen = np.zeros_like(mask_box, dtype=np.uint8)
cv2.drawContours(mask_specimen, [specimen_contour], -1, 255, thickness=cv2.FILLED)

# Opcional: suavizar/cerrar pequeñas irregularidades
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask_specimen = cv2.morphologyEx(mask_specimen, cv2.MORPH_CLOSE, kernel)

# Visualizar contorno sobre la pseudo-RGB
overlay = rgb_u8.copy()
cv2.drawContours(overlay, [specimen_contour], -1, (0, 255, 0), 2)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(mask_box, cmap="gray")
plt.title("Máscara caja azul")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask_specimen, cmap="gray")
plt.title("Máscara espécimen = hueco interno")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Contorno interno detectado")
plt.axis("off")

plt.show()

In [ ]:
specimen_bool = mask_specimen > 0

specimen_cube = cube.copy()
specimen_cube[~specimen_bool, :] = np.nan

spectra_specimen = cube[specimen_bool, :]

print("Píxeles del espécimen:", spectra_specimen.shape[0])
print("Forma espectros:", spectra_specimen.shape)

Código final

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from spectral import open_image


# ============================================================
# 1. Cargar cubo hiperespectral
# ============================================================

hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"

img = open_image(hdr_path)
cube = np.asarray(img.load(), dtype=np.float32)

wavelengths = np.array(
    [float(w) for w in img.metadata["wavelength"]],
    dtype=np.float32
)

print("cube shape:", cube.shape)
print("wavelengths shape:", wavelengths.shape)
print("primeras longitudes de onda:", wavelengths[:10])


# ============================================================
# 2. Funciones auxiliares
# ============================================================

def find_nearest_band(wavelengths, target_nm):
    wavelengths = np.asarray(wavelengths, dtype=float)
    idx = np.argmin(np.abs(wavelengths - target_nm))
    return idx, wavelengths[idx]


def robust_normalize(channel, p_low=2, p_high=98):
    ch = channel.astype(np.float32)
    ch = np.where(np.isfinite(ch), ch, np.nan)

    if np.all(np.isnan(ch)):
        return np.zeros_like(ch, dtype=np.float32)

    lo = np.nanpercentile(ch, p_low)
    hi = np.nanpercentile(ch, p_high)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)

    ch = (ch - lo) / (hi - lo)
    ch = np.clip(ch, 0, 1)
    ch = np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0)

    return ch.astype(np.float32)
def hsi_to_pseudo_rgb(cube, wavelengths, r_nm=650, g_nm=550, b_nm=450):
    r_idx, r_real = find_nearest_band(wavelengths, r_nm)
    g_idx, g_real = find_nearest_band(wavelengths, g_nm)
    b_idx, b_real = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    rgb = np.stack([r, g, b], axis=-1)

    info = {
        "r_idx": r_idx, "r_nm": r_real,
        "g_idx": g_idx, "g_nm": g_real,
        "b_idx": b_idx, "b_nm": b_real
    }

    return rgb, info


# ============================================================
# 3. Generar pseudo-RGB
# ============================================================

pseudo_rgb, rgb_info = hsi_to_pseudo_rgb(
    cube,
    wavelengths,
    r_nm=650,
    g_nm=550,
    b_nm=450
)

print(rgb_info)

pseudo_rgb = np.nan_to_num(pseudo_rgb, nan=0.0, posinf=1.0, neginf=0.0)
rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(rgb_u8)
plt.title(
    f"Pseudo-RGB | "
    f"R={rgb_info['r_nm']:.1f} nm, "
    f"G={rgb_info['g_nm']:.1f} nm, "
    f"B={rgb_info['b_nm']:.1f} nm"
)
plt.axis("off")
plt.show()


# ============================================================
# 4. Detectar la caja azul
# ============================================================

hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

R = rgb_u8[:, :, 0].astype(np.int16)
G = rgb_u8[:, :, 1].astype(np.int16)
B = rgb_u8[:, :, 2].astype(np.int16)

H = hsv[:, :, 0]
S = hsv[:, :, 1]
V = hsv[:, :, 2]

# Rango HSV para azul/cian claro de la caja
# OpenCV usa H en [0, 179]
lower_blue = np.array([85, 10, 40], dtype=np.uint8)
upper_blue = np.array([125, 180, 255], dtype=np.uint8)

mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)

# Condición adicional para evitar blancos/grises:
# el canal azul y verde deben dominar sobre el rojo
mask_dominance = (
    ((B - R) > 8) &
    ((G - R) > 3) &
    (V > 50)
)

mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

# Limpieza de la máscara azul
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

if np.count_nonzero(mask_blue) == 0:
    raise ValueError("No se detectó la caja azul. Relaja los umbrales HSV.")

# Quedarse con el componente conectado más grande
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
    mask_blue,
    connectivity=8
)

if num_labels <= 1:
    raise ValueError("No se encontró ningún componente azul válido.")

areas = stats[1:, cv2.CC_STAT_AREA]
largest_label = 1 + np.argmax(areas)

mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

# Bounding box de la caja azul
x = stats[largest_label, cv2.CC_STAT_LEFT]
y = stats[largest_label, cv2.CC_STAT_TOP]
w = stats[largest_label, cv2.CC_STAT_WIDTH]
h = stats[largest_label, cv2.CC_STAT_HEIGHT]

pad_box = 10
H_img, W_img = rgb_u8.shape[:2]

x1 = max(x - pad_box, 0)
y1 = max(y - pad_box, 0)
x2 = min(x + w + pad_box, W_img)
y2 = min(y + h + pad_box, H_img)

crop_rgb = rgb_u8[y1:y2, x1:x2]
crop_cube = cube[y1:y2, x1:x2, :]

print("BBox caja azul:", x1, y1, x2, y2)
print("crop_rgb shape:", crop_rgb.shape)
print("crop_cube shape:", crop_cube.shape)


# ============================================================
# 5. Usar el hueco interno de la máscara azul como espécimen
# ============================================================

mask_box = mask_box_blue.copy().astype(np.uint8)

contours, hierarchy = cv2.findContours(
    mask_box,
    cv2.RETR_CCOMP,
    cv2.CHAIN_APPROX_SIMPLE
)

if hierarchy is None:
    raise ValueError("No se encontraron contornos en la máscara de la caja.")

hierarchy = hierarchy[0]

# Contornos internos: tienen parent != -1
inner_contours = [
    cnt for cnt, h in zip(contours, hierarchy)
    if h[3] != -1
]

if len(inner_contours) == 0:
    raise ValueError("No se encontró ningún contorno interno en la caja azul.")

# El hueco interno más grande corresponde al espécimen
specimen_contour_original = max(inner_contours, key=cv2.contourArea)

# Crear máscara inicial del espécimen
mask_specimen_original = np.zeros_like(mask_box, dtype=np.uint8)
cv2.drawContours(
    mask_specimen_original,
    [specimen_contour_original],
    -1,
    255,
    thickness=cv2.FILLED
)

# Cerrar pequeñas irregularidades
kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask_specimen_original = cv2.morphologyEx(
    mask_specimen_original,
    cv2.MORPH_CLOSE,
    kernel_smooth
)


# ============================================================
# 6. Agrandar un poco el espécimen para no cortar tejido
# ============================================================

grow_pixels = 5  # prueba 3, 5, 7 según lo conservador que quieras ser

kernel_grow = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (2 * grow_pixels + 1, 2 * grow_pixels + 1)
)

mask_specimen = cv2.dilate(
    mask_specimen_original,
    kernel_grow,
    iterations=1
)

# Recalcular contorno final tras agrandar
contours_final, _ = cv2.findContours(
    mask_specimen,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

if len(contours_final) == 0:
    raise ValueError("No se pudo recalcular el contorno del espécimen.")

specimen_contour = max(contours_final, key=cv2.contourArea)


# ============================================================
# 7. Aplicar máscara al cubo HSI completo
# ============================================================

specimen_bool = mask_specimen > 0

specimen_cube = cube.copy()
specimen_cube[~specimen_bool, :] = np.nan

spectra_specimen = cube[specimen_bool, :]

print("Píxeles del espécimen:", spectra_specimen.shape[0])
print("Forma de spectra_specimen:", spectra_specimen.shape)
print("Forma de specimen_cube:", specimen_cube.shape)


# ============================================================
# 8. También crear recortes útiles
# ============================================================

mask_specimen_crop = mask_specimen[y1:y2, x1:x2]
specimen_bool_crop = mask_specimen_crop > 0

specimen_crop_cube = crop_cube.copy()
specimen_crop_cube[~specimen_bool_crop, :] = np.nan

spectra_specimen_crop = crop_cube[specimen_bool_crop, :]

print("mask_specimen_crop shape:", mask_specimen_crop.shape)
print("specimen_crop_cube shape:", specimen_crop_cube.shape)
print("spectra_specimen_crop shape:", spectra_specimen_crop.shape)


# ============================================================
# 9. Visualización
# ============================================================

# Contorno original
overlay_original = rgb_u8.copy()
cv2.drawContours(
    overlay_original,
    [specimen_contour_original],
    -1,
    (255, 255, 0),
    2
)

# Contorno agrandado
overlay_grow = rgb_u8.copy()
cv2.drawContours(
    overlay_grow,
    [specimen_contour],
    -1,
    (0, 255, 0),
    2
)

# Bounding box de caja azul
box_vis = rgb_u8.copy()
cv2.rectangle(
    box_vis,
    (x1, y1),
    (x2, y2),
    (255, 0, 0),
    2
)

# Visualización en recorte
crop_overlay = crop_rgb.copy()

# El contorno está en coordenadas globales; lo pasamos a coordenadas del crop
specimen_contour_crop = specimen_contour.copy()
specimen_contour_crop[:, 0, 0] -= x1
specimen_contour_crop[:, 0, 1] -= y1

cv2.drawContours(
    crop_overlay,
    [specimen_contour_crop],
    -1,
    (0, 255, 0),
    2
)

masked_vis = rgb_u8.copy()
masked_vis[~specimen_bool] = 0

plt.figure(figsize=(18, 10))

plt.subplot(2, 3, 1)
plt.imshow(rgb_u8)
plt.title("Pseudo-RGB original")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(mask_box_blue, cmap="gray")
plt.title("Máscara caja azul")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(mask_specimen_original, cmap="gray")
plt.title("Máscara espécimen original")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(mask_specimen, cmap="gray")
plt.title(f"Máscara espécimen agrandada +{grow_pixels}px")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(overlay_grow)
plt.title("Contorno final agrandado")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(crop_overlay)
plt.title("Recorte caja azul + contorno final")
plt.axis("off")

plt.tight_layout()
plt.show()

Función final

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False,
    return_info=False,
):
    """
    Carga una imagen hiperespectral ENVI desde una ruta .hdr y devuelve una pseudo-RGB
    donde solo se ve el esp?cimen; el resto queda negro.

    La segmentacion HSI es adaptativa:
        1. Prueba el metodo original basado en caja azul / hueco interno.
        2. Prueba un metodo alternativo para casos donde en gris se ve mejor el tejido
           dentro de la bandeja oscura.
        3. Elige el metodo alternativo solo si el metodo original parece una mascara
           rectangular de la bandeja/caja, y la alternativa tiene calidad suficiente.
    """

    img = open_image(hdr_path)
    import warnings
    try:
        from spectral.io.spyfile import NaNValueWarning
        ignored_warnings = (NaNValueWarning,)
    except Exception:
        ignored_warnings = (Warning,)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ignored_warnings)
        cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array([float(w) for w in img.metadata["wavelength"]], dtype=np.float32)

    def find_nearest_band(wavelengths, target_nm):
        wavelengths_arr = np.asarray(wavelengths, dtype=float)
        return int(np.argmin(np.abs(wavelengths_arr - target_nm)))

    def robust_normalize(channel, p_low=2, p_high=98):
        ch = channel.astype(np.float32)
        ch = np.where(np.isfinite(ch), ch, np.nan)
        if np.all(np.isnan(ch)):
            return np.zeros_like(ch, dtype=np.float32)
        lo = np.nanpercentile(ch, p_low)
        hi = np.nanpercentile(ch, p_high)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            return np.zeros_like(ch, dtype=np.float32)
        ch = (ch - lo) / (hi - lo)
        ch = np.clip(ch, 0, 1)
        return np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)

    def largest_component(mask, min_area=1000, fill=False, avoid_border=False):
        mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
        if num_labels <= 1:
            return np.zeros(mask_u8.shape, dtype=bool)

        img_h, img_w = mask_u8.shape
        candidates = []
        for label in range(1, num_labels):
            x, y, w, h, area = stats[label]
            if area < min_area:
                continue
            if avoid_border and (x <= 2 or y <= 2 or x + w >= img_w - 2 or y + h >= img_h - 2):
                continue
            candidates.append(label)

        if not candidates:
            candidates = [label for label in range(1, num_labels) if stats[label, cv2.CC_STAT_AREA] >= min_area]
        if not candidates:
            candidates = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))]

        cx_img = img_w / 2.0
        cy_img = img_h / 2.0

        def component_score(label):
            x, y, w, h, area = stats[label]
            cx = x + w / 2.0
            cy = y + h / 2.0
            centrality = 1.0 - min(0.8, np.hypot((cx - cx_img) / img_w, (cy - cy_img) / img_h) * 1.5)
            return float(area) * centrality

        best_label = max(candidates, key=component_score)
        selected = labels == best_label

        if fill:
            contours, _ = cv2.findContours(selected.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            filled = np.zeros_like(mask_u8)
            if contours:
                cv2.drawContours(filled, [max(contours, key=cv2.contourArea)], -1, 255, thickness=cv2.FILLED)
            selected = filled > 0

        return selected

    def bbox_xyxy(mask, padding=0):
        ys, xs = np.where(np.asarray(mask) > 0)
        if len(xs) == 0:
            return None
        img_h, img_w = mask.shape[:2]
        return (
            max(int(xs.min()) - padding, 0),
            max(int(ys.min()) - padding, 0),
            min(int(xs.max()) + padding, img_w - 1),
            min(int(ys.max()) + padding, img_h - 1),
        )

    def mask_quality(mask):
        mask_bool = np.asarray(mask) > 0
        ys, xs = np.where(mask_bool)
        if len(xs) == 0:
            return {
                'area_px': 0,
                'bbox_area_px': 0,
                'fill_ratio': 0.0,
                'solidity': 0.0,
                'rectangularity': 0.0,
                'bbox_xyxy': None,
                'area_ratio_image': 0.0,
            }

        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        bbox_area = int((x2 - x1 + 1) * (y2 - y1 + 1))
        area = int(mask_bool.sum())
        fill_ratio = float(area / bbox_area) if bbox_area else 0.0

        contours, _ = cv2.findContours(mask_bool.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        solidity = 0.0
        if contours:
            contour = max(contours, key=cv2.contourArea)
            hull = cv2.convexHull(contour)
            hull_area = cv2.contourArea(hull)
            contour_area = cv2.contourArea(contour)
            solidity = float(contour_area / hull_area) if hull_area > 0 else 0.0

        return {
            'area_px': area,
            'bbox_area_px': bbox_area,
            'fill_ratio': fill_ratio,
            'solidity': solidity,
            'rectangularity': float(fill_ratio * solidity),
            'bbox_xyxy': (x1, y1, x2, y2),
            'area_ratio_image': float(area / mask_bool.size),
        }

    def segment_by_blue_box_inner(rgb_u8):
        hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)
        R = rgb_u8[:, :, 0].astype(np.int16)
        G = rgb_u8[:, :, 1].astype(np.int16)
        B = rgb_u8[:, :, 2].astype(np.int16)
        V = hsv[:, :, 2]

        lower_blue = np.array([85, 10, 40], dtype=np.uint8)
        upper_blue = np.array([125, 180, 255], dtype=np.uint8)
        mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)
        mask_dominance = ((B - R) > 8) & ((G - R) > 3) & (V > 50)
        mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
        kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
        mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
        mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

        if np.count_nonzero(mask_blue) == 0:
            raise ValueError('No se detecto la caja azul. Revisa los umbrales HSV.')

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_blue, connectivity=8)
        if num_labels <= 1:
            raise ValueError('No se encontro ningun componente azul valido.')

        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

        contours, hierarchy = cv2.findContours(mask_box_blue, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
        if hierarchy is None:
            raise ValueError('No se encontraron contornos en la mascara de la caja azul.')
        hierarchy = hierarchy[0]
        inner_contours = [cnt for cnt, h in zip(contours, hierarchy) if h[3] != -1]

        mask_specimen = np.zeros_like(mask_box_blue, dtype=np.uint8)
        source = 'blue_box_inner_contour'

        if inner_contours:
            specimen_contour = max(inner_contours, key=cv2.contourArea)
            cv2.drawContours(mask_specimen, [specimen_contour], -1, 255, thickness=cv2.FILLED)
        else:
            source = 'blue_box_color_fallback'
            x = stats[largest_label, cv2.CC_STAT_LEFT]
            y = stats[largest_label, cv2.CC_STAT_TOP]
            w = stats[largest_label, cv2.CC_STAT_WIDTH]
            h = stats[largest_label, cv2.CC_STAT_HEIGHT]
            pad = 5
            img_h, img_w = rgb_u8.shape[:2]
            x1 = max(int(x) - pad, 0)
            y1 = max(int(y) - pad, 0)
            x2 = min(int(x + w) + pad, img_w)
            y2 = min(int(y + h) + pad, img_h)

            crop_rgb = rgb_u8[y1:y2, x1:x2]
            crop_blue = mask_box_blue[y1:y2, x1:x2] > 0
            crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
            Sc = crop_hsv[:, :, 1]
            Vc = crop_hsv[:, :, 2]
            Rc = crop_rgb[:, :, 0].astype(np.int16)
            Gc = crop_rgb[:, :, 1].astype(np.int16)
            Bc = crop_rgb[:, :, 2].astype(np.int16)

            tissue_color = (((Rc > Bc + 8) | (Gc > Bc + 8) | (Rc > Gc + 8)) & (Sc > 25) & (Vc > 25) & (Vc < 245))
            not_blue_box = ~crop_blue
            not_white_background = ~((Sc < 35) & (Vc > 170))
            not_black_background = Vc > 25
            mask_crop = (tissue_color & not_blue_box & not_white_background & not_black_background).astype(np.uint8) * 255
            mask_crop = cv2.morphologyEx(mask_crop, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
            mask_crop = cv2.morphologyEx(mask_crop, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)))
            mask_crop_clean = largest_component(mask_crop, min_area=1000, fill=True)
            mask_specimen[y1:y2, x1:x2] = mask_crop_clean.astype(np.uint8) * 255

        mask_specimen = cv2.morphologyEx(mask_specimen, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
        return mask_specimen > 0, {'method': source, 'quality': mask_quality(mask_specimen)}

    def segment_by_gray_tray(rgb_u8):
        gray = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        dark = (blur < 55).astype(np.uint8) * 255
        dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (31, 31)))
        tray_roi = largest_component(dark, min_area=20000, fill=True, avoid_border=False)
        roi_bbox = bbox_xyxy(tray_roi, padding=8)
        full_mask = np.zeros_like(gray, dtype=bool)
        if roi_bbox is None:
            return full_mask, {'method': 'gray_tray', 'ok': False, 'reason': 'empty_tray_roi', 'quality': mask_quality(full_mask)}

        x1, y1, x2, y2 = roi_bbox
        roi = tray_roi[y1:y2 + 1, x1:x2 + 1]
        crop_rgb = rgb_u8[y1:y2 + 1, x1:x2 + 1]
        crop_gray = gray[y1:y2 + 1, x1:x2 + 1]
        crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
        S = crop_hsv[:, :, 1]
        V = crop_hsv[:, :, 2]
        R = crop_rgb[:, :, 0].astype(np.int16)
        G = crop_rgb[:, :, 1].astype(np.int16)
        B = crop_rgb[:, :, 2].astype(np.int16)

        vals = crop_gray[roi]
        if vals.size < 100:
            return full_mask, {'method': 'gray_tray', 'ok': False, 'reason': 'small_tray_roi', 'quality': mask_quality(full_mask)}

        threshold = max(15.0, float(np.percentile(vals, 20) + 0.12 * (np.percentile(vals, 98) - np.percentile(vals, 20))))
        chromatic_tissue = (S > 20) & (V > 15) & ((R > B + 2) | (G > B + 2) | (R > G + 2))
        brighter_than_tray = (crop_gray > threshold) & (V > 18)
        candidate = roi & (chromatic_tissue | brighter_than_tray)
        candidate &= ~(V > 225)
        candidate &= ~((S < 25) & (V > 150))

        candidate = cv2.morphologyEx(candidate.astype(np.uint8) * 255, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
        candidate = cv2.erode(candidate, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=1)
        candidate = largest_component(candidate, min_area=5000, fill=False, avoid_border=True)
        candidate = cv2.dilate(candidate.astype(np.uint8) * 255, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1)
        candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (51, 51)))
        candidate = largest_component(candidate, min_area=5000, fill=True, avoid_border=False)
        candidate = cv2.morphologyEx(candidate.astype(np.uint8) * 255, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))) > 0
        candidate = largest_component(candidate, min_area=5000, fill=True, avoid_border=False)

        full_mask[y1:y2 + 1, x1:x2 + 1] = candidate
        info = {
            'method': 'gray_tray',
            'ok': True,
            'roi_bbox_xyxy': roi_bbox,
            'threshold': float(threshold),
            'quality': mask_quality(full_mask),
        }
        return full_mask, info

    r_idx = find_nearest_band(wavelengths, r_nm)
    g_idx = find_nearest_band(wavelengths, g_nm)
    b_idx = find_nearest_band(wavelengths, b_nm)

    pseudo_rgb = np.stack([
        robust_normalize(cube[:, :, r_idx]),
        robust_normalize(cube[:, :, g_idx]),
        robust_normalize(cube[:, :, b_idx]),
    ], axis=-1)
    rgb_u8 = (np.clip(np.nan_to_num(pseudo_rgb, nan=0.0, posinf=1.0, neginf=0.0), 0, 1) * 255).astype(np.uint8)

    blue_mask, blue_info = segment_by_blue_box_inner(rgb_u8)
    gray_mask, gray_info = segment_by_gray_tray(rgb_u8)

    blue_q = blue_info['quality']
    gray_q = gray_info['quality']
    blue_looks_rectangular = (
        blue_info['method'] == 'blue_box_inner_contour'
        and blue_q['fill_ratio'] >= 0.82
        and blue_q['solidity'] >= 0.90
        and blue_q['area_ratio_image'] >= 0.12
    )
    gray_is_usable = (
        gray_info.get('ok', False)
        and gray_q['area_px'] >= 5000
        and gray_q['fill_ratio'] >= 0.25
        and gray_q['fill_ratio'] <= 0.92
        and gray_q['solidity'] >= 0.45
    )
    gray_is_better_shape = gray_q['rectangularity'] <= blue_q['rectangularity'] - 0.10

    if blue_looks_rectangular and gray_is_usable and gray_is_better_shape:
        chosen_mask = gray_mask
        chosen_info = gray_info.copy()
        chosen_info['selected_because'] = 'blue_mask_looked_rectangular'
    else:
        chosen_mask = blue_mask
        chosen_info = blue_info.copy()
        chosen_info['selected_because'] = 'blue_mask_quality_ok'

    chosen_info = {
        'selected_method': chosen_info.get('method'),
        'selected_because': chosen_info.get('selected_because'),
        'blue_box_candidate': blue_info,
        'gray_tray_candidate': gray_info,
        'wavelengths_nm': {
            'r': float(wavelengths[r_idx]),
            'g': float(wavelengths[g_idx]),
            'b': float(wavelengths[b_idx]),
        },
    }

    mask_specimen = chosen_mask.astype(np.uint8) * 255
    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * grow_pixels + 1, 2 * grow_pixels + 1))
        mask_specimen = cv2.dilate(mask_specimen, kernel_grow, iterations=1)

    specimen_only_rgb = rgb_u8.copy()
    specimen_only_rgb[mask_specimen == 0] = 0

    if show_original and show_result:
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis('off')
        plt.subplot(1, 2, 2)
        plt.imshow(specimen_only_rgb)
        plt.title(f"Solo especimen | metodo: {chosen_info['selected_method']}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis('off')
        plt.show()
    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(specimen_only_rgb)
        plt.title(f"Solo especimen | metodo: {chosen_info['selected_method']}")
        plt.axis('off')
        plt.show()

    if return_mask and return_info:
        return specimen_only_rgb, mask_specimen, chosen_info
    if return_mask:
        return specimen_only_rgb, mask_specimen
    if return_info:
        return specimen_only_rgb, chosen_info
    return specimen_only_rgb


In [ ]:
hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm_edu.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(specimen_only_rgb)
plt.title("Specimen only RGB")
plt.axis("off")
plt.show()

In [ ]:
hdr_path = r"Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm_edu.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)

In [ ]:
hdr_path = r"Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_nrm_edu.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)

In [ ]:
hdr_path = r"Datos\SB018\SB018\SB018_001\SB018_nrm_edu.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)   

In [ ]:
hdr_path = r"Datos\SB019\SB019\SB019_001\SB019_nrm_edu.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)  

In [ ]:
hdr_path = r"Datos\SB020\SB020\SB020_001\SB020_nrm_edu.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)

In [ ]:
hdr_path = r"Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_raw.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)